# Échantillonnage stratifié du dataset Cifer Fraud Detection

Objectif : réduire 21M lignes à un échantillon exploitable sur 8 Go de RAM,
en gardant 100% des fraudes et un sous-échantillon aléatoire des non-fraudes.
 
Stratégie :
  - Lecture fichier par fichier (jamais les 14 CSV en mémoire en même temps)
  - Downcast des types dès la lecture (float64->float32, int64->int32/int8)
  - Séparation fraude / non-fraude à la volée, libération mémoire immédiate
  - Sous-échantillonnage aléatoire de la non-fraude (seed fixée = reproductible)
  - Ratio cible configurable (défaut : 50 non-fraudes pour 1 fraude)
 
Auteur : Rasmané

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import gc

## 0. CONFIGURATION

In [ ]:
DOSSIER_DATA = r"C:\Users\hp\Documents\Fraude_detection\data"
CHEMIN_SORTIE = os.path.join(DOSSIER_DATA, "Cifer-echantillon-strat.csv")
 
RATIO_NON_FRAUDE_PAR_FRAUDE = 50   # 50:1 -> à ajuster si besoin
SEED = 42                          # reproductibilité (exigence du sujet, point 18)
 
rng = np.random.default_rng(SEED)
 
# Types optimisés pour réduire l'empreinte mémoire dès la lecture
DTYPES = {
    "step": "int16",
    "type": "category",
    "amount": "float32",
    "nameOrig": "string",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "nameDest": "string",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
}

## 1. LOCALISER LES FICHIERS

In [ ]:
motif = os.path.join(DOSSIER_DATA, "Cifer-Fraud-Detection-Dataset-AF-part-*.csv")
liste_fichiers = sorted(
    glob.glob(motif),
    key=lambda x: int(x.split("part-")[1].split("-")[0])
)
print(f"Nombre de fichiers trouvés : {len(liste_fichiers)}")
 
if len(liste_fichiers) == 0:
    raise FileNotFoundError("Aucun fichier trouvé. Vérifie DOSSIER_DATA et le motif.")

## 2. PASSE 1 : EXTRAIRE TOUTES LES FRAUDES + COMPTER LES NON-FRAUDES

In [ ]:
# On garde toutes les fraudes en mémoire (elles sont rares, ~27 500 lignes
# au total sur 21M -> volume négligeable).
# Pour la non-fraude, on fait un échantillonnage réservoir approximatif :
# on tire un sous-échantillon aléatoire à même chaque fichier, proportionnel
# à sa taille, pour ne jamais charger toute la non-fraude en mémoire.
 
liste_fraudes = []
liste_non_fraudes_echantillon = []
total_non_fraude_vu = 0
 
for i, fichier in enumerate(liste_fichiers, start=1):
    print(f"\nTraitement fichier {i}/{len(liste_fichiers)} : {os.path.basename(fichier)}")
 
    df = pd.read_csv(fichier, dtype=DTYPES)
    df["fichier_source"] = os.path.basename(fichier)
 
    # Séparer fraude / non-fraude dans ce fichier
    df_fraude = df[df["isFraud"] == 1]
    df_non_fraude = df[df["isFraud"] == 0]
 
    print(f"  Lignes : {len(df):,} | Fraudes : {len(df_fraude)} | "
          f"Non-fraudes : {len(df_non_fraude):,}")
 
    liste_fraudes.append(df_fraude.copy())
    total_non_fraude_vu += len(df_non_fraude)
 
    # Échantillonnage aléatoire immédiat de la non-fraude de CE fichier
    # (on suréchantillonne légèrement ici, le calibrage final se fait après
    # la passe 1 une fois qu'on connaît le nombre total de fraudes)
    taille_echantillon_fichier = min(len(df_non_fraude), 200_000)
    if len(df_non_fraude) > 0:
        echantillon = df_non_fraude.sample(
            n=taille_echantillon_fichier, random_state=SEED
        )
        liste_non_fraudes_echantillon.append(echantillon.copy())
 
    # Libération mémoire explicite avant de passer au fichier suivant
    del df, df_fraude, df_non_fraude
    gc.collect()

## 3. ASSEMBLER LES FRAUDES

In [ ]:
data_fraude = pd.concat(liste_fraudes, ignore_index=True)
del liste_fraudes
gc.collect()
 
nb_fraudes_total = len(data_fraude)
print(f"\n{'='*70}")
print(f"Total fraudes collectées : {nb_fraudes_total:,}")
print(f"Total non-fraudes vues (tous fichiers) : {total_non_fraude_vu:,}")

## 4. CALIBRER L'ÉCHANTILLON DE NON-FRAUDE AU RATIO CIBLE

In [ ]:
nb_non_fraude_cible = nb_fraudes_total * RATIO_NON_FRAUDE_PAR_FRAUDE
print(f"Nombre de non-fraudes cible (ratio {RATIO_NON_FRAUDE_PAR_FRAUDE}:1) : "
      f"{nb_non_fraude_cible:,}")
 
data_non_fraude_brut = pd.concat(liste_non_fraudes_echantillon, ignore_index=True)
del liste_non_fraudes_echantillon
gc.collect()
 
print(f"Non-fraudes déjà pré-échantillonnées disponibles : {len(data_non_fraude_brut):,}")
 
if len(data_non_fraude_brut) >= nb_non_fraude_cible:
    # On a assez -> tirage final exact au ratio voulu
    data_non_fraude = data_non_fraude_brut.sample(
        n=int(nb_non_fraude_cible), random_state=SEED
    )
else:
    # Pas assez (cas rare si ratio très élevé) -> on garde tout ce qu'on a
    print("ATTENTION : pas assez de non-fraudes pré-échantillonnées pour")
    print("atteindre le ratio cible. Augmente taille_echantillon_fichier")
    print("dans la passe 1, ou réduis RATIO_NON_FRAUDE_PAR_FRAUDE.")
    data_non_fraude = data_non_fraude_brut
 
del data_non_fraude_brut
gc.collect()

## 5. ASSEMBLER L'ÉCHANTILLON FINAL

In [ ]:
data_echantillon = pd.concat([data_fraude, data_non_fraude], ignore_index=True)
# Mélanger les lignes (sinon toutes les fraudes sont regroupées au début)
data_echantillon = data_echantillon.sample(frac=1, random_state=SEED).reset_index(drop=True)
 
del data_fraude, data_non_fraude
gc.collect()
 
print(f"\n{'='*70}")
print("ÉCHANTILLON FINAL")
print(f"{'='*70}")
print(f"Lignes totales : {len(data_echantillon):,}")
print(f"Fraudes : {data_echantillon['isFraud'].sum():,}")
print(f"Non-fraudes : {(data_echantillon['isFraud']==0).sum():,}")
ratio_final = (data_echantillon['isFraud']==0).sum() / data_echantillon['isFraud'].sum()
print(f"Ratio final non-fraude:fraude : {ratio_final:.1f} : 1")
print(f"Taux de fraude dans l'échantillon : {data_echantillon['isFraud'].mean()*100:.3f}%")
 
# Empreinte mémoire de l'échantillon final
memoire_mo = data_echantillon.memory_usage(deep=True).sum() / 1024**2
print(f"Empreinte mémoire de l'échantillon : {memoire_mo:.1f} Mo")

## 6. SAUVEGARDE

In [ ]:
data_echantillon.to_csv(CHEMIN_SORTIE, index=False)
print(f"\nÉchantillon sauvegardé : {CHEMIN_SORTIE}")

## 7. NOTE MÉTHODOLOGIQUE À REPRENDRE DANS TON RAPPORT

In [ ]:
print(f"""
{'='*70}
NOTE MÉTHODOLOGIQUE (à copier/adapter dans ton rapport, section limites)
{'='*70}
Stratégie d'échantillonnage : échantillonnage stratifié asymétrique.
100% des transactions frauduleuses ont été conservées ({nb_fraudes_total:,}
lignes), car elles constituent la classe rare porteuse du signal principal.
Un sous-échantillon aléatoire (seed={SEED}, reproductible) de la classe
majoritaire a été tiré pour atteindre un ratio de {RATIO_NON_FRAUDE_PAR_FRAUDE}:1,
contre environ 763.5:1 dans le dataset
d'origine (valeur calculée lors de l'EDA initiale). Ce choix est motivé par une contrainte matérielle (8 Go de RAM)
tout en préservant un déséquilibre de classe réaliste et représentatif du
problème métier (~2% de fraude dans l'échantillon final vs <0.15% dans
l'ensemble complet). Le tirage aléatoire de la non-fraude a été effectué
fichier par fichier avant agrégation afin de représenter l'ensemble de la
période temporelle simulée (30 jours) et non uniquement une portion.
""")
 